# Laboratorium 4 (4 pkt.)

Celem czwartego laboratorium jest zapoznanie się oraz zaimplementowanie algorytmów głębokiego uczenia aktywnego. Zaimplementowane algorytmy będą testowane z wykorzystaniem wcześniej przygotowanych środowisk: *FrozenLake* i *Pacman* oraz środowiska z OpenAI - *CartPole*.


Dołączenie standardowych bibliotek

In [1]:
from collections import deque
import gymnasium as gym
import numpy as np
import random
import os

Dołączenie bibliotek ze środowiskami:

In [2]:
from env.FrozenLakeMDP import frozenLake
from env.FrozenLakeMDPExtended import frozenLakeExtended


Dołączenie bibliotek do obsługi sieci neuronowych

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(342)

class DQN(nn.Module):
    def __init__(self, state_size, action_size, hidden_neurons, num_layers=1, learning_rate=0.001):
        super(DQN, self).__init__()

        self.entrance = nn.Linear(state_size, hidden_neurons)
        for i in range(num_layers - 1):
             setattr(self, f'fc{i+1}', nn.Linear(hidden_neurons, hidden_neurons))
        self.out = nn.Linear(hidden_neurons, action_size)

        self.learning_rate = learning_rate
        self.optimizer = optim.AdamW(self.parameters(), lr=self.learning_rate)

    def forward(self, x):
        x = F.relu(self.entrance(x))
        for i in range(len(self._modules) - 2):
            x = F.relu(getattr(self, f'fc{i+1}')(x))

        return self.out(x)
    
    def predict(self, state):
        state = torch.FloatTensor(state)
        with torch.no_grad():
            q_values = self.forward(state)
        
        return q_values.numpy()
    
    def fit(self, states, targets):
        if len(states) == 0:
            return
        
        states = torch.tensor(states)
        targets = torch.tensor(targets)
        
        if states.ndim == 1:
            states = states.reshape(1, -1)
        if targets.ndim == 1:
            targets = targets.reshape(1, -1)

        states = torch.FloatTensor(states)
        targets = torch.FloatTensor(targets)

        self.optimizer.zero_grad()
        outputs = self.forward(states)
        loss = F.smooth_l1_loss(outputs, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.parameters(), 1.0)
        self.optimizer.step()

## Zadanie 1 - Deep Q-Network

<p style='text-align: justify;'>
Celem ćwiczenie jest zaimplementowanie algorytmu Deep Q-Network. Wartoscią oczekiwaną sieci jest:
\begin{equation}
        Q(s_t, a_t) = r_{t+1} + \gamma \text{max}_a Q(s_{t + 1}, a)
\end{equation}
</p>

In [4]:
np.random.seed(342)

class DQNAgent:
    def __init__(self, action_size, learning_rate, model):
        self.action_size = action_size
        self.memory = deque(maxlen=2000)
        self.gamma = 0.95    # discount rate
        self.epsilon = 1.0  # exploration rate
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.95
        self.learning_rate = learning_rate
        self.model = model

    def remember(self, state, action, reward, next_state, done):
        #Function adds information to the memory about last action and its results
        self.memory.append((state, action, reward, next_state, done)) 

    def get_action(self, state):
        """
        Compute the action to take in the current state, including exploration.
        With probability self.epsilon, we should take a random action.
            otherwise - the best policy action (self.get_best_action).

        Note: To pick randomly from a list, use random.choice(list).
              To pick True or False with a given probablity, generate uniform number in [0, 1]
              and compare it with your probability
        """

        if random.uniform(0, 1) < self.epsilon:
            chosen_action = random.choice(range(self.action_size))
        else:
            chosen_action = self.get_best_action(state)

        
        return chosen_action

  
    def get_best_action(self, state):
        """
        Compute the best action to take in a state.
        """
        q_values = self.model.predict(state) 
        
        best_value = np.max(q_values)
        best_actions = np.where(q_values == best_value)[0]
        best_action = random.choice(best_actions)

        return best_action

    def replay(self, batch_size):
        """
        Function learn network using randomly selected actions from the memory. 
        First calculates Q value for the next state and choose action with the biggest value.
        Target value is calculated according to:
                Q(s,a) := (r + gamma * max_a(Q(s', a)))
        except the situation when the next action is the last action, in such case Q(s, a) := r.
        In order to change only those weights responsible for chosing given action, the rest values should be those
        returned by the network for state state.
        The network should be trained on batch_size samples.
        """
    
        if len(self.memory) < batch_size:
            return

        mini_batch = random.sample(self.memory, batch_size)

        states = []
        targets = []

        for state, action, reward, next_state, done in mini_batch:

            target_f = np.array(self.model.predict(state), copy=True)

            if done:
                target = reward
            else:
                next_q = np.max(self.model.predict(next_state))
                target = reward + self.gamma * next_q

            target_f[action] = target

            states.append(state.numpy())
            targets.append(target_f)

        states = np.array(states, dtype=np.float32)
        targets = np.array(targets, dtype=np.float32)

        self.model.fit(states, targets)

    def update_epsilon_value(self):
        #Every each epoch epsilon value should be updated according to equation: 
        #self.epsilon *= self.epsilon_decay, but the updated value shouldn't be lower then epsilon_min value
        
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def save_model(self, epoch, path):
        if not os.path.exists(path):
            os.makedirs(path)

        torch.save(self.model.state_dict(), f'{path}/dqn_model_epoch_{epoch}.pth')

    def load_model(self, epoch, path):
        self.model.load_state_dict(torch.load(f'{path}/dqn_model_epoch_{epoch}.pth'))

Czas przygotować model sieci, która będzie się uczyła poruszania po środowisku *FrozenLake*, warstwa wejściowa powinna mieć tyle neuronów ile jest możlliwych stanów, warstwa wyjściowa tyle neuronów ile jest możliwych akcji do wykonania:

In [5]:
env = frozenLake("8x8")

state_size = env.get_number_of_states()
action_size = len(env.get_possible_actions(None))
learning_rate = 0.001

print("State size: {}, Action size: {}".format(state_size, action_size))

model = DQN(state_size, action_size, 32, 2, learning_rate=learning_rate)

State size: 64, Action size: 4


 Czas nauczyć agenta poruszania się po środowisku *FrozenLake*, jako stan przyjmij wektor o liczbie elementów równej liczbie możliwych stanów, z wartością 1 ustawioną w komórce o indeksie równym aktualnemu stanowi, pozostałe elementy mają być wypełnione zerami:
* 1 pkt < 35 epok,
* 0.5 pkt < 60 epok,
* 0.25 pkt - w pozostałych przypadkach.

In [6]:
agent = DQNAgent(action_size, learning_rate, model)

agent.epsilon = 0.75

done = False
batch_size = 64
EPISODES = 10000
counter = 0
for e in range(EPISODES):

    summary = []
    for _ in range(100):
        total_reward = 0
        env_state = env.reset()

    
        state = torch.zeros(state_size)
        state[env_state] = 1
        
        for time in range(1000):
            action = agent.get_action(state)
            next_state_env, reward, done, _ = env.step(action)
            total_reward += reward

            next_state = torch.zeros(state_size) 
            next_state[next_state_env] = 1

            agent.remember(state, action, reward, next_state, done)
            state = next_state
            if done:
                break
            
            if len(agent.memory) > batch_size and time % 10 == 0:
                agent.replay(batch_size)
                
        summary.append(total_reward)
    agent.update_epsilon_value()

    print("epoch #{}\tmean reward = {:.3f}\tepsilon = {:.3f}".format(e, np.mean(summary), agent.epsilon))
    if np.mean(summary) > 0.9:
        print ("You Win!")
        break

# agent.save_model(e, 'frozen_lake_models/8x8')

IndexError: Dimension out of range (expected to be in range of [-1, 0], but got 1)

Czas przygotować model sieci, która będzie się uczyła poruszania po środowisku *FrozenLakeExtended*, tym razem stan nie jest określany poprzez pojedynczą liczbę, a przez 3 tablice:
* pierwsza zawierająca informacje o celu,
* druga zawierająca informacje o dziurach,
* trzecia zawierająca informację o położeniu gracza.

In [ ]:
env = frozenLakeExtended("4x4")

state_size = env.get_number_of_states()
action_size = len(env.get_possible_actions(None))
learning_rate = 0.001

model = DQN(state_size * 3, action_size, 32, 2, learning_rate)

 Czas nauczyć agenta poruszania się po środowisku *FrozenLakeExtended*, jako stan przyjmij wektor składający się ze wszystkich trzech tablic (2 pkt.):

In [ ]:
agent = DQNAgent(action_size, learning_rate, model)

agent.epsilon = 0.75

done = False
batch_size = 64
EPISODES = 2000
counter = 0
for e in range(EPISODES):
    summary = []
    for _ in range(100):
        total_reward = 0
        env_state = env.reset()

        state = [np.asarray(part).ravel() for part in env_state]
        state = np.concatenate(state).astype(float)
        state = torch.tensor(state, dtype=torch.float32)
        
        for time in range(1000):
            action = agent.get_action(state)
            next_state_env, reward, done, _ = env.step(action)
            total_reward += reward

            next_state = [np.asarray(part).ravel() for part in next_state_env]
            next_state = np.concatenate(next_state).astype(float)
            next_state = torch.tensor(next_state, dtype=torch.float32)

            #add to experience memory
            agent.remember(state, action, reward, next_state, done)
            state = next_state
            if done:
                break

            if len(agent.memory) > batch_size and time % 10 == 0:
                agent.replay(batch_size)
                
        summary.append(total_reward)
    agent.update_epsilon_value()

    if np.mean(summary) > 0.9:
        print ("You Win!")
        break
    print("epoch #{}\tmean reward = {:.3f}\tepsilon = {:.3f}".format(e, np.mean(summary), agent.epsilon))

    # agent.save_model(e, 'frozen_lake_models/4x4')

Czas przygotować model sieci, która będzie się uczyła działania w środowisku [*CartPool*](https://gym.openai.com/envs/CartPole-v0/):

In [ ]:
env = gym.make("CartPole-v0").env
state_size = env.observation_space.shape[0]
action_size = env.action_space.n
learning_rate = 0.001

print("State size: {}, Action size: {}".format(state_size, action_size))

model = DQN(state_size, action_size, 24, 2, learning_rate)

Czas nauczyć agenta gry w środowisku *CartPool*:
* 1 pkt < 10 epok,
* 0.5 pkt < 20 epok,
* 0.25 pkt - w pozostałych przypadkach.

In [ ]:
agent = DQNAgent(action_size, learning_rate, model)

agent.epsilon = 0.75

done = False
batch_size = 128
EPISODES = 1000
counter = 0
for e in range(EPISODES):
    summary = []
    for _ in range(100):
        total_reward = 0
        env_state = env.reset()[0]

        state = torch.tensor(env_state, dtype=torch.float32)

        for time in range(300):
            action = agent.get_action(state)
            next_state_env, reward, done, _, _ = env.step(action)
            total_reward += reward

            next_state = torch.tensor(next_state_env, dtype=torch.float32)

            agent.remember(state, action, reward, next_state, done)
            state = next_state
            if done:
                break
            
            if len(agent.memory) > batch_size and time % 5 == 0:
                agent.replay(batch_size)
                
        summary.append(total_reward)
    agent.update_epsilon_value()

    if np.mean(summary) > 195:
        print ("You Win!")
        break
    print("epoch #{}\tmean reward = {:.3f}\tepsilon = {:.3f}".format(e, np.mean(summary), agent.epsilon))

    # agent.save_model(e, 'cartpole_models')